In [ ]:
# Kaggle cell
!pip install -q "diffusers==0.30.3" scikit-learn seaborn


## 2. Configuration

`DATA_ROOT` can be `/kaggle/input` because the helper below searches nested Kaggle dataset folders. If automatic discovery fails, point it directly at the directory containing category folders.


In [ ]:
import os
import json
import math
import random
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
from sklearn.metrics import roc_auc_score, average_precision_score

# -------- Change these first --------
DATA_ROOT = "/kaggle/input"
CATEGORY = "bottle"

# Full improved run. Start with the smoke-test values below if needed.
IMAGE_SIZE = 256
BATCH_SIZE = 8                 # global batch; try 16 if both GPUs have room
EPOCHS = 75
INFER_STEPS = 50
N_INPAINT_MASKS = 9            # 3 x 3 tiles cover the image
INPAINT_SAMPLES = 2            # use 3 for the final, slower evaluation

# Smoke-test alternative: EPOCHS=5, INFER_STEPS=10, INPAINT_SAMPLES=1
LEARNING_RATE = 1e-4
NUM_TRAIN_TIMESTEPS = 1000
AUX_X0_WEIGHT = 0.10           # auxiliary masked clean-image L1 loss
NUM_WORKERS = 2
TOP_FRACTION = 0.01
MASK_BORDER = 6
FOREGROUND_THRESHOLD = 0.985

RUN_TRAINING = True
MAX_TEST_IMAGES = None         # set e.g. 20 for a quick evaluation
RUN_RESIDUAL_CALIBRATION = False
CALIBRATION_MAX_ITEMS = None   # use 40 for a quick diagnostic; None for all normals

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = OUT_DIR / f"mvtec_{CATEGORY}_diffusion_inpaint_improved.pt"

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
print("device:", DEVICE)
print("visible GPUs:", torch.cuda.device_count())
print("output:", OUT_DIR)
if Path("/kaggle/input").exists():
    print("Kaggle inputs:", [p.name for p in Path("/kaggle/input").iterdir()])


## 3. Locate and load MVTec-AD

The loader uses normal images from `train/good`. For test images it also reads the corresponding `ground_truth/<defect>/*_mask.png` file.


In [ ]:
IMG_EXTENSIONS = {".png", ".jpg", ".jpeg"}

def find_mvtec_root(root, category):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")

    candidates = [root]
    candidates.extend([p for p in root.iterdir() if p.is_dir()])
    for candidate in candidates:
        if (candidate / category / "train").is_dir() and (candidate / category / "test").is_dir():
            return candidate

    # More general search for nested Kaggle layouts.
    for train_dir in root.rglob("train"):
        if train_dir.parent.name == category and (train_dir.parent / "test").is_dir():
            return train_dir.parent.parent

    raise FileNotFoundError(
        f"Could not find <root>/{category}/train and test below {root}. "
        f"Set DATA_ROOT to the folder containing category folders."
    )

ROOT = find_mvtec_root(DATA_ROOT, CATEGORY)
CATEGORY_ROOT = ROOT / CATEGORY
print("MVTec root:", ROOT)
print("Category:", CATEGORY_ROOT)

train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
eval_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
mask_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.NEAREST),
    T.ToTensor(),
])

class MVTecDataset(Dataset):
    def __init__(self, root, category, split="train", transform=None):
        self.base = Path(root) / category
        self.split = split
        self.transform = transform
        self.items = []

        if split == "train":
            image_dir = self.base / "train" / "good"
            for path in sorted(image_dir.iterdir()):
                if path.suffix.lower() in IMG_EXTENSIONS:
                    self.items.append((path, None, 0))
        elif split == "test":
            test_dir = self.base / "test"
            for defect_dir in sorted(p for p in test_dir.iterdir() if p.is_dir()):
                defect = defect_dir.name
                for path in sorted(defect_dir.iterdir()):
                    if path.suffix.lower() not in IMG_EXTENSIONS:
                        continue
                    if defect == "good":
                        gt_path, label = None, 0
                    else:
                        gt_path = self.base / "ground_truth" / defect / f"{path.stem}_mask{path.suffix}"
                        # Some copies of MVTec use PNG masks even if the image extension changes.
                        if not gt_path.exists():
                            gt_path = self.base / "ground_truth" / defect / f"{path.stem}_mask.png"
                        label = 1
                    self.items.append((path, gt_path, label))
        else:
            raise ValueError("split must be train or test")

        if not self.items:
            raise RuntimeError(f"No images found for {category}/{split} under {self.base}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        image_path, gt_path, label = self.items[index]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        if gt_path is None or not gt_path.exists():
            gt = torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
        else:
            gt_image = Image.open(gt_path).convert("L")
            gt = (mask_transform(gt_image) > 0.5).float()

        return image, gt, torch.tensor(label, dtype=torch.long), str(image_path)

train_ds = MVTecDataset(ROOT, CATEGORY, split="train", transform=train_transform)
train_eval_ds = MVTecDataset(ROOT, CATEGORY, split="train", transform=eval_transform)
test_ds = MVTecDataset(ROOT, CATEGORY, split="test", transform=eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)
train_eval_loader = DataLoader(
    train_eval_ds, batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)
test_loader = DataLoader(
    test_ds, batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)

print(f"train normal images: {len(train_ds)}")
print(f"test images: {len(test_ds)}")


In [ ]:
# Quick visual check of the data.
images, masks, labels, paths = next(iter(test_loader))
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow((images[0].permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1))
axes[0].set_title(Path(paths[0]).parent.name)
axes[1].imshow(masks[0, 0], cmap="gray")
axes[1].set_title("ground-truth mask")
axes[2].axis("off")
axes[0].axis("off")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 4. Synthetic hole masks

The model never sees an anomalous training image. It learns inpainting from random rectangles and occasional irregular masks cut out of normal images. A value of 1 means that a pixel is hidden and should be generated.


In [ ]:
def random_inpaint_masks(batch_size, height, width, device):
    masks = torch.zeros(batch_size, 1, height, width, device=device)

    for i in range(batch_size):
        n_rectangles = random.randint(1, 4)
        for _ in range(n_rectangles):
            if random.random() < 0.55:
                # Small holes model localized MVTec defects.
                rh = random.randint(max(6, height // 32), max(8, height // 4))
                rw = random.randint(max(6, width // 32), max(8, width // 4))
            else:
                # Larger holes model structural inpainting.
                rh = random.randint(max(16, height // 8), max(17, height // 2))
                rw = random.randint(max(16, width // 8), max(17, width // 2))

            y0 = random.randint(0, max(0, height - rh))
            x0 = random.randint(0, max(0, width - rw))
            masks[i, 0, y0:y0 + rh, x0:x0 + rw] = 1.0

        if random.random() < 0.40:
            low_h, low_w = max(4, height // 16), max(4, width // 16)
            coarse = torch.rand(1, 1, low_h, low_w, device=device)
            irregular = F.interpolate(
                coarse, size=(height, width), mode="bilinear", align_corners=False
            ) > 0.52
            masks[i] = torch.maximum(masks[i], irregular.float()[0])

        if masks[i].sum() == 0:
            masks[i, 0, height // 4:height // 2, width // 4:width // 2] = 1.0

    return masks

def make_eval_masks(n_masks, height, width, device):
    patch = max(16, int(round(0.50 * min(height, width))))
    y_starts = sorted(set([0, (height - patch) // 2, height - patch]))
    x_starts = sorted(set([0, (width - patch) // 2, width - patch]))
    masks = []
    for y0 in y_starts:
        for x0 in x_starts:
            m = torch.zeros(1, height, width, device=device)
            m[:, y0:y0 + patch, x0:x0 + patch] = 1.0
            masks.append(m)

    target_n = max(n_masks, len(masks))
    while len(masks) < target_n:
        rh = random.randint(max(16, height // 3), max(17, height * 2 // 3))
        rw = random.randint(max(16, width // 3), max(17, width * 2 // 3))
        y0 = random.randint(0, max(0, height - rh))
        x0 = random.randint(0, max(0, width - rw))
        m = torch.zeros(1, height, width, device=device)
        m[:, y0:y0 + rh, x0:x0 + rw] = 1.0
        masks.append(m)

    output = torch.stack(masks[:target_n], dim=0)
    assert output.sum(dim=0).min() > 0, "evaluation masks do not cover the whole image"
    return output

def estimate_foreground(image, threshold=FOREGROUND_THRESHOLD):
    """Approximate the object mask from the near-white MVTec background."""
    rgb = (image * 0.5 + 0.5).clamp(0.0, 1.0)
    gray = rgb.mean(dim=1, keepdim=True)
    foreground = (gray < threshold).float()
    # A small dilation makes the condition channel less fragile at antialiased edges.
    foreground = F.max_pool2d(foreground, kernel_size=11, stride=1, padding=5)
    return foreground

def erode_binary_mask(mask, radius=6):
    """Keep only pixels sufficiently inside a binary mask."""
    if radius <= 0:
        return mask
    kernel = 2 * radius + 1
    pooled = F.avg_pool2d(mask, kernel_size=kernel, stride=1, padding=radius)
    return (pooled > 0.999).float()

example_mask = random_inpaint_masks(1, IMAGE_SIZE, IMAGE_SIZE, DEVICE)[0, 0].cpu()
plt.figure(figsize=(3, 3))
plt.imshow(example_mask, cmap="gray")
plt.title("synthetic training hole")
plt.axis("off")
plt.show()


## 5. Wider foreground-conditioned diffusion model

The U-Net receives noisy RGB, the hole mask, visible RGB context, and a near-white-background foreground mask. The extra foreground channel helps it distinguish the industrial object from the white background.

The model returns a tensor rather than a Diffusers `ModelOutput`, allowing `torch.nn.DataParallel` to use two GPUs safely.


In [ ]:
class NoisePredictionUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.unet = UNet2DModel(
            sample_size=IMAGE_SIZE,
            in_channels=8,  # noisy RGB + hole + visible RGB + foreground
            out_channels=3,
            layers_per_block=2,
            block_out_channels=(96, 192, 384, 768),
            down_block_types=("DownBlock2D", "DownBlock2D","DownBlock2D", "AttnDownBlock2D"),
            up_block_types=("AttnUpBlock2D", "UpBlock2D","UpBlock2D", "UpBlock2D"),
            norm_num_groups=32,
            act_fn="silu",
        )

    def forward(self, sample, timestep):
        # Return a tensor so DataParallel can gather outputs safely.
        return self.unet(sample, timestep).sample

model = NoisePredictionUNet().to(DEVICE)
GPU_IDS = list(range(torch.cuda.device_count())) if DEVICE.type == "cuda" else []
if len(GPU_IDS) > 1:
    model = nn.DataParallel(model, device_ids=GPU_IDS, output_device=GPU_IDS[0])
    print(f"using DataParallel on GPUs: {GPU_IDS}")
else:
    print(f"using one device: {DEVICE}")

def unwrap_model(module):
    return module.module if isinstance(module, nn.DataParallel) else module

num_parameters = sum(p.numel() for p in model.parameters())
print(f"parameters: {num_parameters / 1e6:.2f}M")

train_scheduler = DDPMScheduler(
    num_train_timesteps=NUM_TRAIN_TIMESTEPS,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon",
    clip_sample=False,
)
sample_scheduler = DDIMScheduler.from_config(train_scheduler.config)
ALPHAS_CUMPROD = train_scheduler.alphas_cumprod.to(DEVICE)


## 6. Train on normal images only

The main objective predicts diffusion noise. An auxiliary masked L1 loss on the predicted clean image encourages the generated hole to match the normal target instead of only minimizing noise-prediction error.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.99), weight_decay=1e-4
)
lr_schedule = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, EPOCHS)
)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def save_checkpoint(epoch, history):
    torch.save(
        {
            "epoch": epoch,
            "model": unwrap_model(model).state_dict(),
            "optimizer": optimizer.state_dict(),
            "history": history,
            "category": CATEGORY,
            "image_size": IMAGE_SIZE,
            "in_channels": 8,
        },
        CHECKPOINT,
    )

def train_model():
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        start = time.time()
        batch = 1

        for images, _, _, _ in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            batch_size = images.shape[0]
            masks = random_inpaint_masks(
                batch_size, IMAGE_SIZE, IMAGE_SIZE, DEVICE
            )
            visible = images * (1.0 - masks)
            foreground = estimate_foreground(images)

            timesteps = torch.randint(
                0,
                NUM_TRAIN_TIMESTEPS,
                (batch_size,),
                device=DEVICE,
                dtype=torch.long,
            )
            noise = torch.randn_like(images)
            noisy = train_scheduler.add_noise(images, noise, timesteps)
            model_input = torch.cat(
                [noisy, masks, visible, foreground], dim=1
            )

            optimizer.zero_grad(set_to_none=True)
            amp_context = (
                torch.autocast(device_type="cuda", dtype=torch.float16)
                if USE_AMP else nullcontext()
            )
            with amp_context:
                predicted_noise = model(model_input, timesteps)

                noise_loss = (
                    (predicted_noise - noise).pow(2)
                    * (0.25 + 0.75 * masks)
                ).mean()

                # Convert epsilon prediction into an x0 estimate and add a
                # direct masked reconstruction objective.
                alpha_bar = ALPHAS_CUMPROD[timesteps].view(-1, 1, 1, 1)
                predicted_x0 = (
                    noisy
                    - torch.sqrt(1.0 - alpha_bar) * predicted_noise
                ) / torch.sqrt(alpha_bar)
                predicted_x0 = predicted_x0.clamp(-1.0, 1.0)

                masked_l1 = (
                    (predicted_x0 - images).abs() * masks
                ).sum() / (3.0 * masks.sum() + 1e-6)

                loss = noise_loss + AUX_X0_WEIGHT * masked_l1

            if USE_AMP:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            running_loss += loss.item()
            print(f"Epoch [{epoch}] | Batch [{batch}] | Loss: {loss:.4f}")
            batch += 1

        lr_schedule.step()
        epoch_loss = running_loss / max(1, len(train_loader))
        history.append(epoch_loss)
        save_checkpoint(epoch, history)
        print(
            f"epoch {epoch:03d}/{EPOCHS} | loss {epoch_loss:.5f} | "
            f"lr {optimizer.param_groups[0]['lr']:.2e} | "
            f"{time.time() - start:.1f}s"
        )
    return history

if RUN_TRAINING:
    history = train_model()
else:
    if not CHECKPOINT.exists():
        raise FileNotFoundError(
            f"No checkpoint at {CHECKPOINT}; set RUN_TRAINING=True"
        )
    checkpoint = torch.load(CHECKPOINT, map_location=DEVICE)
    unwrap_model(model).load_state_dict(checkpoint["model"])
    history = checkpoint.get("history", [])

plt.figure(figsize=(6, 3))
plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("noise + masked x0 loss")
plt.title("training curve")
plt.grid(alpha=0.25)
plt.show()
print("checkpoint:", CHECKPOINT)


## 7. Diffusion inpainting sampler

Known pixels are re-injected at every reverse step. The sampler averages several stochastic inpainting samples per mask, which reduces random residual noise. The timestep is expanded to the batch dimension because `DataParallel` cannot scatter a scalar timestep.


In [ ]:
@torch.no_grad()
def add_noise_at(scheduler, clean, noise, timestep):
    t_value = int(timestep.item()) if torch.is_tensor(timestep) else int(timestep)
    t = torch.full(
        (clean.shape[0],),
        t_value,
        device=clean.device,
        dtype=torch.long,
    )
    return scheduler.add_noise(clean, noise, t)

@torch.no_grad()
def inpaint_sample(model, scheduler, image, hole_mask, steps=50):
    model.eval()
    scheduler.set_timesteps(steps, device=image.device)

    known_noise = torch.randn_like(image)
    sample = torch.randn_like(image)
    first_t = scheduler.timesteps[0]
    known_at_first = add_noise_at(
        scheduler, image, known_noise, first_t
    )
    sample = (
        sample * hole_mask
        + known_at_first * (1.0 - hole_mask)
    )
    visible = image * (1.0 - hole_mask)
    foreground = estimate_foreground(image)

    for step_index, timestep in enumerate(scheduler.timesteps):
        model_input = torch.cat(
            [sample, hole_mask, visible, foreground],
            dim=1,
        )

        # DataParallel cannot scatter a scalar timestep tensor.
        model_timestep = torch.full(
            (sample.shape[0],),
            int(timestep.item())
            if torch.is_tensor(timestep)
            else int(timestep),
            device=sample.device,
            dtype=torch.long,
        )

        amp_context = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if USE_AMP else nullcontext()
        )
        with amp_context:
            predicted_noise = model(model_input, model_timestep)

        sample_prev = scheduler.step(
            predicted_noise,
            timestep,
            sample,
            eta=0.0,
        ).prev_sample

        if step_index + 1 < len(scheduler.timesteps):
            next_t = scheduler.timesteps[step_index + 1]
            known_next = add_noise_at(
                scheduler, image, known_noise, next_t
            )
        else:
            known_next = image

        sample = (
            sample_prev * hole_mask
            + known_next * (1.0 - hole_mask)
        )

    return sample.clamp(-1.0, 1.0)

@torch.no_grad()
def diffusion_anomaly_map(
    image,
    n_masks=N_INPAINT_MASKS,
    steps=INFER_STEPS,
    n_samples=INPAINT_SAMPLES,
):
    # image is [1, 3, H, W] in [-1, 1].
    _, _, height, width = image.shape
    masks = make_eval_masks(
        n_masks, height, width, image.device
    )
    repeated_image = image.repeat(masks.shape[0], 1, 1, 1)

    residual_sum = torch.zeros(
        masks.shape[0], 1, height, width,
        device=image.device,
    )
    reconstruction_sum = torch.zeros_like(repeated_image)

    for _ in range(n_samples):
        reconstructions = inpaint_sample(
            model,
            sample_scheduler,
            repeated_image,
            masks,
            steps=steps,
        )
        reconstruction_sum += reconstructions
        residual_sum += (
            repeated_image - reconstructions
        ).abs().mean(dim=1, keepdim=True)

    residual = residual_sum / float(n_samples)
    reconstructions = reconstruction_sum / float(n_samples)

    # Score only pixels sufficiently inside a hole and on the object.
    score_masks = erode_binary_mask(
        masks, radius=MASK_BORDER
    )
    foreground = estimate_foreground(image).repeat(
        masks.shape[0], 1, 1, 1
    )
    score_masks = score_masks * foreground

    weighted = (residual * score_masks).sum(
        dim=0, keepdim=True
    )
    coverage = score_masks.sum(dim=0, keepdim=True)
    anomaly_map = weighted / (coverage + 1e-6)
    anomaly_map = F.avg_pool2d(
        anomaly_map,
        kernel_size=5,
        stride=1,
        padding=2,
    )
    return anomaly_map.squeeze(0), masks, reconstructions

# One qualitative smoke-test sample.
model.eval()

INFER_STEPS = 30
N_INPAINT_MASKS = 9
INPAINT_SAMPLES = 1

with torch.no_grad():
    sample_image = images[:1].to(DEVICE)
    anomaly_map, used_masks, sample_reconstructions = diffusion_anomaly_map(
        sample_image,
        n_masks=N_INPAINT_MASKS,
        steps=INFER_STEPS,
        n_samples=INPAINT_SAMPLES,
    )

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
original = (
    sample_image[0].detach().cpu().permute(1, 2, 0).numpy()
    * 0.5 + 0.5
).clip(0, 1)
reconstructed = (
    sample_reconstructions[0].detach().cpu().permute(1, 2, 0).numpy()
    * 0.5 + 0.5
).clip(0, 1)
axes[0].imshow(original); axes[0].set_title("input")
axes[1].imshow(used_masks[0, 0].cpu(), cmap="gray"); axes[1].set_title("one hole mask")
axes[2].imshow(reconstructed); axes[2].set_title("mean inpainting")
axes[3].imshow(anomaly_map[0].cpu(), cmap="magma"); axes[3].set_title("residual map")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 8. Evaluate image- and pixel-level anomaly detection

The anomaly score is the mean of the largest `TOP_FRACTION` pixels in the diffusion residual map. This is less diluted by a small defect than a whole-image mean.


In [ ]:
@torch.no_grad()
def collect_scores(loader, max_items=None):
    model.eval()
    scores, labels, maps, gt_masks, original_images, paths = [], [], [], [], [], []
    seen = 0

    for batch_images, batch_gt, batch_labels, batch_paths in loader:
        batch_images = batch_images.to(DEVICE, non_blocking=True)
        for j in range(batch_images.shape[0]):
            image = batch_images[j:j + 1]
            amap, _, _ = diffusion_anomaly_map(
                image, n_masks=N_INPAINT_MASKS, steps=INFER_STEPS
            )
            flat = amap.flatten()
            k = max(1, int(flat.numel() * TOP_FRACTION))
            score = torch.topk(flat, k=k).values.mean().item()

            scores.append(score)
            labels.append(int(batch_labels[j].item()))
            maps.append(amap[0].detach().cpu().numpy())
            gt_masks.append(batch_gt[j, 0].numpy())
            original_images.append(batch_images[j].detach().cpu())
            paths.append(batch_paths[j])
            seen += 1

            # if seen % 10 == 0:
            print(f"scored {seen} images")
            if max_items is not None and seen >= max_items:
                return {
                    "scores": np.asarray(scores), "labels": np.asarray(labels),
                    "maps": maps, "gt_masks": gt_masks,
                    "images": original_images, "paths": paths,
                }

    return {
        "scores": np.asarray(scores), "labels": np.asarray(labels),
        "maps": maps, "gt_masks": gt_masks,
        "images": original_images, "paths": paths,
    }

def safe_auc(y_true, y_score):
    return float(roc_auc_score(y_true, y_score)) if len(np.unique(y_true)) == 2 else float("nan")

# Full test evaluation can take time because it samples several masks per image.
test_results = collect_scores(test_loader, max_items=None)
image_auc = safe_auc(test_results["labels"], test_results["scores"])

pixel_true = np.concatenate([m.reshape(-1) for m in test_results["gt_masks"]])
pixel_score = np.concatenate([m.reshape(-1) for m in test_results["maps"]])
pixel_auc = safe_auc(pixel_true, pixel_score)
pixel_ap = (
    float(average_precision_score(pixel_true, pixel_score))
    if pixel_true.max() > pixel_true.min() else float("nan")
)

print(f"image ROC-AUC: {image_auc:.4f}")
print(f"pixel ROC-AUC: {pixel_auc:.4f}")
print(f"pixel average precision: {pixel_ap:.4f}")


In [ ]:
def plot_top_results(results, n=6):
    n = min(n, len(results["scores"]))
    indices = np.argsort(results["scores"])[-n:][::-1]
    fig, axes = plt.subplots(n, 4, figsize=(13, max(3, 3 * n)))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(indices):
        image = (results["images"][idx].permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
        amap = results["maps"][idx]
        gt = results["gt_masks"][idx]

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"input | score={results['scores'][idx]:.4f}")
        axes[row, 1].imshow(amap, cmap="magma")
        axes[row, 1].set_title("diffusion residual")
        axes[row, 2].imshow(image)
        axes[row, 2].imshow(amap, cmap="jet", alpha=0.45)
        axes[row, 2].set_title("overlay")
        axes[row, 3].imshow(gt, cmap="gray")
        axes[row, 3].set_title("ground truth")
        for col in range(4):
            axes[row, col].axis("off")

    plt.tight_layout()
    output_path = OUT_DIR / f"{CATEGORY}_qualitative.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("saved:", output_path)

plot_top_results(test_results, n=3)
